In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder

def apply_business_logic(df, variance_threshold=1.5):
    """
    Applies corporate risk rules to the Machine Learning confidence intervals to
    generate a final, firm quote for the customer.

    The logic evaluates the spread between the median prediction (P50) and the
    conservative prediction (P90). If the spread is large, it indicates high network
    volatility, and the system quotes the safer P90 value to protect the SLA. If
    the network is stable, it quotes the faster P50 value to remain competitive.

    Parameters
    ----------
    df : pandas.DataFrame
        The prediction dataset output from Phase 3. Must contain 'Pred_P50_Days'
        and 'Pred_P90_Days' columns.
    variance_threshold : float, optional (default=1.5)
        The maximum acceptable difference in days between P50 and P90 before the
        system triggers a conservative (P90) quote.

    Returns
    -------
    df : pandas.DataFrame
        The dataframe with a new column: 'AI_Quoted_Transit_Days'.
    """
    print(f"Applying business logic (Risk Variance Threshold: {variance_threshold} days)...")

    # Calculate the variance spread
    df['Risk_Spread'] = df['Pred_P90_Days'] - df['Pred_P50_Days']

    # Apply logic: Use P90 if spread > threshold, else use P50
    # np.ceil is used to round up to the next whole day (you cannot quote 3.2 days to a customer)
    df['AI_Quoted_Transit_Days'] = np.where(
        df['Risk_Spread'] > variance_threshold,
        np.ceil(df['Pred_P90_Days']),
        np.ceil(df['Pred_P50_Days'])
    )

    return df


def run_quoting_simulation(df, actual_col):
    """
    Simulates historical network performance as if the company had been using the
    AI Quoting tool instead of the static matrix. Evaluates the improvement in
    Net Service Level (NSL).

    Parameters
    ----------
    df : pandas.DataFrame
        The dataset containing the historical actuals, the old quoted days, and
        the new AI quoted days.
    actual_col : str
        The name of the column containing the actual time the shipment took
        (e.g., 'Calculated_Actual_Days' or 'Actual_Transit_Days').

    Returns
    -------
    df : pandas.DataFrame
        The dataframe with the new boolean outcome 'AI_Delivered_in_Commit'.
    """
    print("\nRunning historical A/B simulation...")

    # Determine simulated success: Did the package arrive within the new AI quoted window?
    df['AI_Delivered_in_Commit'] = (df[actual_col] <= df['AI_Quoted_Transit_Days']).astype(int)

    # 1. Historical Metrics (What actually happened)
    # Re-calculate historical NSL based on the original data's actual vs quoted to ensure accuracy
    if 'Quoted_Transit_Days' in df.columns:
        df['Historical_Delivered_in_Commit'] = (df[actual_col] <= df['Quoted_Transit_Days']).astype(int)
        historical_nsl = df['Historical_Delivered_in_Commit'].mean() * 100
        avg_hist_quote = df['Quoted_Transit_Days'].mean()
    else:
        # Fallback if Quoted_Transit_Days was dropped during feature engineering
        historical_nsl = df['Delivered_in_Commit'].mean() * 100
        avg_hist_quote = 0.0 # Placeholder

    # 2. Simulated AI Metrics (What would have happened)
    simulated_nsl = df['AI_Delivered_in_Commit'].mean() * 100
    avg_ai_quote = df['AI_Quoted_Transit_Days'].mean()

    # Calculate impact
    nsl_improvement = simulated_nsl - historical_nsl
    padding_increase = avg_ai_quote - avg_hist_quote

    print("\n" + "="*45)
    print(" === CAPSTONE A/B SIMULATION RESULTS ===")
    print("="*45)
    print(f"Historical (Static Matrix) NSL : {historical_nsl:.2f}%")
    print(f"Optimized AI-Driven Quoting NSL: {simulated_nsl:.2f}%")
    print("-" * 45)
    print(f"Net NSL Improvement            : +{nsl_improvement:.2f}%")
    print("-" * 45)
    print(f"Avg Historical Quoted Days     : {avg_hist_quote:.2f} days")
    print(f"Avg AI Quoted Days             : {avg_ai_quote:.2f} days")
    print(f"Average Buffer Added per Load  : +{padding_increase:.2f} days")
    print("=============================================\n")

    return df


def generate_postal_adjustment_report(df):
    """
    Groups the simulation results by destination postal code and pickup day of the week
    to identify exactly which postal codes require transit time adjustments,
    on which days, and by how many days.

    Parameters
    ----------
    df : pandas.DataFrame
        The finalized simulated dataset containing both historical quotes and
        the newly generated AI quotes.

    Returns
    -------
    postal_summary : pandas.DataFrame
        A dataframe summarizing the average days added per postal code and day of week,
        sorted by the locations needing the most drastic adjustments.
    """
    print("Generating Postal Code & Day of Week Adjustment Report...")

    if 'Quoted_Transit_Days' not in df.columns or 'dest_pstl_cd' not in df.columns:
        print("[WARNING] Missing required columns to generate postal adjustment report.")
        return None

    # Create the grouping columns
    group_cols = ['dest_pstl_cd']

    # Safely decode the Day of Week if it exists from Phase 3
    if 'Pickup_Day_of_Week' in df.columns:
        day_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
        df['Pickup_Day_Name'] = df['Pickup_Day_of_Week'].map(day_map).fillna(df['Pickup_Day_of_Week'])
        group_cols.append('Pickup_Day_Name')

    # Calculate how many days the AI added/subtracted for each specific shipment
    df['Days_Adjusted'] = df['AI_Quoted_Transit_Days'] - df['Quoted_Transit_Days']

    # Group by destination postal code (and Day of Week)
    postal_summary = df.groupby(group_cols).agg(
        Total_Shipments=('AI_Quoted_Transit_Days', 'count'),
        Historical_Avg_Quote=('Quoted_Transit_Days', 'mean'),
        AI_Avg_Quote=('AI_Quoted_Transit_Days', 'mean'),
        Avg_Days_Added=('Days_Adjusted', 'mean')
    ).reset_index()

    # Round metrics for clean reporting
    postal_summary['Historical_Avg_Quote'] = postal_summary['Historical_Avg_Quote'].round(1)
    postal_summary['AI_Avg_Quote'] = postal_summary['AI_Avg_Quote'].round(1)
    postal_summary['Avg_Days_Added'] = postal_summary['Avg_Days_Added'].round(1)

    # Sort to show the biggest bottlenecks (most days added) at the top
    postal_summary = postal_summary.sort_values(by=['Avg_Days_Added', 'Total_Shipments'], ascending=[False, False])

    print("\n=== TOP 10 POSTAL CODES REQUIRING TRANSIT TIME INCREASES (BY DAY) ===")
    print(postal_summary.head(10).to_string(index=False))
    print("=======================================================================\n")

    return postal_summary


def forecast_manual_buffer_impact(df, postal_summary, actual_col, top_n=5, buffer_options=[1, 2]):
    """
    Simulates the "What-If" operational scenario of manually adding buffer days
    to the historical static quotes for only the worst-performing postal codes.

    Parameters
    ----------
    df : pandas.DataFrame
        The finalized simulated dataset containing historical quotes and actual transit days.
    postal_summary : pandas.DataFrame
        The generated report of worst-performing postal codes (from generate_postal_adjustment_report).
    actual_col : str
        The name of the column containing the actual time the shipment took.
    top_n : int, optional (default=5)
        The number of top worst postal codes to apply the manual buffer to.
    buffer_options : list, optional (default=[1, 2])
        A list of integers representing the number of days to test adding to the quote.

    Returns
    -------
    None
        Outputs the forecasting metrics directly to the console.
    """
    if postal_summary is None or 'Quoted_Transit_Days' not in df.columns:
        return

    print(f"Forecasting NSL impact of manually buffering the Top {top_n} worst postal codes...")

    # Extract the unique postal codes that appeared at the very top of the bottleneck report
    worst_postals = postal_summary['dest_pstl_cd'].unique()[:top_n]
    print(f"Target Problem Postals: {', '.join(map(str, worst_postals))}")

    # Baseline NSL for comparison
    historical_nsl = (df[actual_col] <= df['Quoted_Transit_Days']).mean() * 100
    print(f"Baseline Historical NSL: {historical_nsl:.2f}%")

    for buffer in buffer_options:
        # Copy the original quotes to simulate a manual system adjustment
        hypothetical_quotes = df['Quoted_Transit_Days'].copy()

        # Apply the buffer ONLY to shipments destined for the top N problem postals
        mask = df['dest_pstl_cd'].isin(worst_postals)
        hypothetical_quotes.loc[mask] += buffer

        # Calculate the new hypothetical NSL
        hypothetical_nsl = (df[actual_col] <= hypothetical_quotes).mean() * 100
        improvement = hypothetical_nsl - historical_nsl

        print(f" - If +{buffer} Day(s) added to Top {top_n}: NSL jumps to {hypothetical_nsl:.2f}% (Improvement: +{improvement:.2f}%)")

    print("=======================================================================\n")


# --- Execution Block ---
if __name__ == "__main__":
    # Define file paths
    input_file = "Phase3_Predictions.csv"
    input_path = os.path.join("../..", "Data", "Processed", input_file)
    output_path = os.path.join("../..", "Data", "Processed", "Phase4_Final_Simulation.csv")
    postal_report_path = os.path.join("../..", "Data", "Processed", "Phase4_Postal_Adjustments.csv")
    original_data_path = os.path.join("../..", "Data", "Processed", "Cleaned_IEF_Shipments_FY26.csv")

    try:
        # 1. Load the predictions from Phase 3
        print(f"Loading model predictions from {input_path}...")
        results_df = pd.read_csv(input_path)

        # --- FIX: DECODE POSTAL CODES ---
        # The postal codes were Label Encoded in Phase 3 (0, 1, 2, etc.).
        # We recreate the encoder from the original data to decode them back to real zip codes.
        if os.path.exists(original_data_path):
            print("Decoding postal codes back to their original string values...")
            original_df = pd.read_csv(original_data_path)
            le = LabelEncoder()
            le.fit(original_df['dest_pstl_cd'].astype(str))

            # Inverse transform the encoded numbers back to zip codes
            results_df['dest_pstl_cd'] = le.inverse_transform(results_df['dest_pstl_cd'])
        else:
            print(f"[WARNING] Original data not found at {original_data_path}.")
            print("Postal codes will remain as encoded integers.")
        # --------------------------------

        # Safely identify the 'Actual' column name
        target_col = 'Calculated_Actual_Days' if 'Calculated_Actual_Days' in results_df.columns else 'Actual_Transit_Days'

        # 2. Apply the risk threshold business logic
        applied_logic_df = apply_business_logic(results_df, variance_threshold=1.5)

        # 3. Run the final A/B Simulation (Overall Metrics)
        final_df = run_quoting_simulation(applied_logic_df, target_col)

        # 4. Generate the specific Postal Code Adjustment Report
        postal_adjustments_df = generate_postal_adjustment_report(final_df)

        # 5. NEW: Run the NSL impact forecast for manual adjustments to top bottleneck postals
        forecast_manual_buffer_impact(final_df, postal_adjustments_df, target_col, top_n=5, buffer_options=[1, 2])

        # 6. Save the final datasets
        final_df.to_csv(output_path, index=False)
        print(f"Final simulated dataset saved to {output_path} successfully.")

        if postal_adjustments_df is not None:
            postal_adjustments_df.to_csv(postal_report_path, index=False)
            print(f"Postal code adjustment report saved to {postal_report_path} successfully.")

    except FileNotFoundError:
        print(f"[ERROR] Could not find {input_path}.")
        print("Ensure you have run 'phase3_modeling.py' to generate the predictions first.")

Loading model predictions from ..\Data\Processed\Phase3_Predictions.csv...
Decoding postal codes back to their original string values...
Applying business logic (Risk Variance Threshold: 1.5 days)...

Running historical A/B simulation...

 === CAPSTONE A/B SIMULATION RESULTS ===
Historical (Static Matrix) NSL : 68.77%
Optimized AI-Driven Quoting NSL: 87.10%
---------------------------------------------
Net NSL Improvement            : +18.33%
---------------------------------------------
Avg Historical Quoted Days     : 7.41 days
Avg AI Quoted Days             : 10.42 days
Average Buffer Added per Load  : +3.01 days

Generating Postal Code & Day of Week Adjustment Report...

=== TOP 10 POSTAL CODES REQUIRING TRANSIT TIME INCREASES (BY DAY) ===
dest_pstl_cd Pickup_Day_Name  Total_Shipments  Historical_Avg_Quote  AI_Avg_Quote  Avg_Days_Added
        3019         Tuesday                2                   4.0          10.0             6.0
        3019       Wednesday                1     